Training for Main Project

Part 1: Data Cleaning

In [1]:
import pandas as pd
import numpy as np
import glob
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load all CSVs into one DataFrame
path = 'hawkeye/play_by_play'
all_files = glob.glob(path + "/*.csv")

df_list = []
for filename in all_files:
    df = pd.read_csv(filename, index_col=None, header=0)
    df_list.append(df)

master_df = pd.concat(df_list, axis=0, ignore_index=True)
print(f"Total raw serves loaded: {len(master_df)}")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
import torch
import torch.nn as nn

class ServeClassifierLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes=3):
        super(ServeClassifierLSTM, self).__init__()
        # The LSTM layer processes the sequence step-by-step
        self.lstm = nn.LSTM(input_size=input_dim, 
                            hidden_size=hidden_dim, 
                            batch_first=True)
        
        # Dense Layer + Softmax (handled implicitly by CrossEntropyLoss)
        self.fc = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        # x shape: (batch_size, 3, num_features)
        # out shape: (batch_size, 3, hidden_dim)
        # h_n shape: (1, batch_size, hidden_dim) -> final hidden states
        out, (h_n, c_n) = self.lstm(x)
        
        # Extract the hidden state of the very last sequence step (Many-to-One)
        last_step_out = out[:, -1, :] # shape: (batch_size, hidden_dim)
        
        # Pass through the Dense layer
        logits = self.fc(last_step_out)
        return logits